# 6. Fine-tuning CLAP on the other setups

In `4_new_setup.ipynb` CLAP reads the family with text prompts and no training: 0.43 on average on a setup it has
never seen. Here we train its sound encoder on the labeled datasets we have.

Does it get better on a setup that was **not** in the training?

This notebook is slow (several hours on a GPU). Each experiment saves its table in `results/` and is skipped if the
table is already there.

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from sklearn.metrics import balanced_accuracy_score
from transformers import ClapModel, ClapProcessor

FAMILIES = ["wood", "metal", "textile", "polymer"]
PUBLIC = ["LMT108", "LMT69", "TUM", "SENS3", "Cluster", "Greatest Hits"]
device = "cuda"

saved = np.load("cache/sound_windows.npz", allow_pickle=True)
windows = saved["windows"]
info = pd.DataFrame({c: saved[c] for c in ["dataset", "recording", "surface", "family"]})

model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

## Same reading as in notebook 4

Sentences for each family, one vector per recording, vectors centred on the tested dataset.

In [ ]:
templates = [
    "the sound of scraping {}",
    "a stick scratching a {} surface",
    "rubbing a finger on {}",
    "friction noise of sliding on {}",
    "scratching {}",
]
words = {
    "wood": ["wood", "a wooden board", "timber"],
    "metal": ["metal", "steel", "aluminium"],
    "textile": ["fabric", "cloth", "textile"],
    "polymer": ["plastic", "rubber", "vinyl"],
}

text_vectors = []
for family in FAMILIES:
    sentences = [t.format(w) for t in templates for w in words[family]]
    inputs = processor(text=sentences, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        out = model.get_text_features(**inputs).pooler_output
    mean = torch.nn.functional.normalize(out, dim=-1).mean(0)
    text_vectors.append(mean / mean.norm())
text_vectors = torch.stack(text_vectors)


def sound_vectors(rows):
    # CLAP vectors of some windows, with the weights the model has now
    batch = torch.tensor(windows[rows].astype(np.float32)) * 0.1
    batch = torchaudio.functional.resample(batch, 16000, 48000).numpy()
    inputs = processor(audio=list(batch), sampling_rate=48000, return_tensors="pt").to(device)
    return torch.nn.functional.normalize(model.get_audio_features(**inputs).pooler_output, dim=-1)


def balanced_draws(answers):
    # some families have many more surfaces: we draw the same number in each, 30 times
    rng = np.random.default_rng(0)
    surfaces = answers.drop_duplicates("surface")
    families = [f for f in FAMILIES if f in set(answers.family)]
    n = surfaces.family.value_counts().min()
    scores = []
    for _ in range(30):
        kept = np.concatenate([rng.choice(surfaces.surface[surfaces.family == f], n, replace=False) for f in families])
        part = answers[answers.surface.isin(kept)]
        scores.append(balanced_accuracy_score(part.family, part.answer))
    return np.mean(scores)


def score(rows):
    # balanced accuracy of the text reading on these windows (all from one dataset)
    with torch.no_grad():
        vectors = torch.cat([sound_vectors(rows[i : i + 32]) for i in range(0, len(rows), 32)]).cpu().numpy()
    part = info.iloc[rows]
    per_recording = pd.DataFrame(vectors).groupby(part.recording.to_numpy()).mean()
    answers = part.drop_duplicates("recording").set_index("recording").loc[per_recording.index]
    families = [f for f in FAMILIES if f in set(answers.family)]

    v = per_recording.to_numpy() - per_recording.to_numpy().mean(axis=0)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    cosine = v @ text_vectors.cpu().numpy()[[FAMILIES.index(f) for f in families]].T
    answers["answer"] = np.array(families)[(cosine - cosine.mean(axis=0)).argmax(axis=1)]
    return balanced_draws(answers)

## The training

We train the whole sound encoder (the text side stays frozen) so that each window is close to the sentences of
its family. 1200 steps, 12 windows per step, the same number from every (dataset, family) pair.
Each training dataset also gets its own small bias, so the model does not have to learn which family is the most
frequent in which dataset.

In [ ]:
original = {k: v.clone() for k, v in model.state_dict().items()}
trained_parts = list(model.audio_model.parameters()) + list(model.audio_projection.parameters())
CHECKS = [0, 25, 50, 100, 200, 400, 800, 1200]


def set_weights(tuned, share=1.0):
    # share = 0: original CLAP, share = 1: fine-tuned, in between: a mix of both
    mixed = {}
    for k in original:
        if original[k].is_floating_point():
            mixed[k] = (1 - share) * original[k] + share * tuned[k]
        else:
            mixed[k] = original[k]
    model.load_state_dict(mixed)


def fine_tune(train_rows, test_rows=None, rate=2e-5, steps=1200, half=None):
    # returns the fine-tuned weights and, if test_rows is given, the score on them along the training
    # half = name of a dataset that must fill half of every batch (used in notebook 7)
    set_weights(original)
    for p in model.parameters():
        p.requires_grad_(False)
    for p in trained_parts:
        p.requires_grad_(True)

    part = info.iloc[train_rows]
    sources = sorted(part.dataset.unique())
    groups = {}
    for d in sources:
        for f in FAMILIES:
            rows = train_rows[((part.dataset == d) & (part.family == f)).to_numpy()]
            if len(rows):
                groups[d, f] = rows
    keys = list(groups)

    bias = torch.zeros(len(sources), len(FAMILIES), device=device, requires_grad=True)
    optimizer = torch.optim.AdamW([{"params": trained_parts, "lr": rate}, {"params": [bias], "lr": 1e-2}], weight_decay=0.01)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=[rate, 1e-2], total_steps=steps, pct_start=0.1)
    scale = float(model.logit_scale_a.exp())
    rng = np.random.default_rng(0)
    curve = []
    for step in range(steps + 1):
        if test_rows is not None and step in CHECKS:
            curve.append(score(test_rows))
        if step == steps:
            break
        if half:
            first = [key for key in keys if key[0] == half]
            rest = [key for key in keys if key[0] != half]
            picked = [first[i] for i in rng.integers(0, len(first), 6)] + [rest[i] for i in rng.integers(0, len(rest), 6)]
        else:
            picked = [keys[i] for i in rng.integers(0, len(keys), 12)]
        rows = np.array([rng.choice(groups[key]) for key in picked])
        truth = torch.tensor([FAMILIES.index(f) for _, f in picked], device=device)
        source = torch.tensor([sources.index(d) for d, _ in picked], device=device)
        logits = scale * sound_vectors(rows) @ text_vectors.T + bias[source]
        loss = torch.nn.functional.cross_entropy(logits, truth)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        schedule.step()
    return {k: v.clone() for k, v in model.state_dict().items()}, curve


def rows_of(datasets):
    return np.flatnonzero(info.dataset.isin(datasets).to_numpy())


def others(target):
    # LMT108 and LMT69 have the same surfaces: when one is tested, the other is not used for training
    twins = {"LMT108": "LMT69", "LMT69": "LMT108"}
    return [d for d in PUBLIC if d != target and d != twins.get(target)]

## Experiment 1: leave one dataset out

Each public dataset is left out in turn. We train on the others and follow the score of the left-out one during
the training. We do it with the normal learning rate and with one 10 times lower.

At the end we also mix the fine-tuned weights with the original ones (25, 50, 75 %).

In [ ]:
if not Path("results/fine_tuning_curve.csv").exists():
    curve_rows, mix_rows = [], []
    for target in PUBLIC:
        test = rows_of([target])
        for rate in [2e-5, 2e-6]:
            tuned, curve = fine_tune(rows_of(others(target)), test, rate)
            curve_rows += [{"rate": rate, "target": target, "step": s, "score": v} for s, v in zip(CHECKS, curve)]
            if rate == 2e-5:
                torch.save(tuned, f"cache/clap_without_{target}.pt")
                for share in [0, 0.25, 0.5, 0.75, 1]:
                    set_weights(tuned, share)
                    mix_rows.append({"target": target, "share": share, "score": score(test)})
    pd.DataFrame(curve_rows).to_csv("results/fine_tuning_curve.csv", index=False)
    pd.DataFrame(mix_rows).to_csv("results/fine_tuning_mixing.csv", index=False)

curve = pd.read_csv("results/fine_tuning_curve.csv")
curve[curve.rate == 2e-5].pivot(index="target", columns="step", values="score").round(2)

In [ ]:
mean = curve.groupby(["rate", "step"]).score.mean().unstack("rate")
plt.figure(figsize=(6, 3.5))
for rate in mean.columns:
    plt.plot(range(len(mean)), mean[rate], marker="o", label=f"learning rate {rate:g}")
plt.axhline(mean.iloc[0].mean(), color="green", linestyle="--", label="original CLAP")
plt.xticks(range(len(mean)), mean.index)
plt.xlabel("training step on the other datasets")
plt.ylabel("score on the left-out dataset")
plt.legend()
plt.tight_layout()
plt.savefig("results/fine_tuning_curve.png", dpi=120)

In [ ]:
mixing = pd.read_csv("results/fine_tuning_mixing.csv")
mixing.pivot(index="target", columns="share", values="score").round(2)

The score of the left-out dataset goes down from the first steps and never comes back above its start.
A lower learning rate only makes the fall slower. Mixing the weights gives a point between the two models, never
above the original.

## Experiment 2: does it at least work on setups it has seen?

We keep 30 % of the surfaces of every dataset aside and train on the rest of the six datasets together.
For comparison we also train on each dataset alone.

In [ ]:
if not Path("results/fine_tuning_known_setups.csv").exists():
    surfaces = info[info.dataset.isin(PUBLIC)].drop_duplicates(["dataset", "surface"])
    held_out = surfaces.groupby(["dataset", "family"]).sample(frac=0.3, random_state=0)
    is_test = (info.dataset + "|" + info.surface).isin(held_out.dataset + "|" + held_out.surface).to_numpy()
    is_public = info.dataset.isin(PUBLIC).to_numpy()

    rows = []
    merged, _ = fine_tune(np.flatnonzero(is_public & ~is_test))
    for d in PUBLIC:
        test = np.flatnonzero((info.dataset == d).to_numpy() & is_test)
        set_weights(original)
        row = {"dataset": d, "original CLAP": score(test)}
        set_weights(merged)
        row["six datasets together"] = score(test)
        alone, _ = fine_tune(np.flatnonzero((info.dataset == d).to_numpy() & ~is_test))
        row["this dataset alone"] = score(test)
        rows.append(row)
    pd.DataFrame(rows).to_csv("results/fine_tuning_known_setups.csv", index=False)

known = pd.read_csv("results/fine_tuning_known_setups.csv", index_col=0)
known.loc["mean"] = known.mean()
known.round(2)

Fine-tuning works very well on new surfaces of a setup it has seen. And one dataset alone does as well as the six
together: the other setups do not help, even here.

## Experiment 3: more setups in the training

Maybe five setups are not enough variety. We cannot add more, but we can remove some: for each left-out dataset
we train on 1, 2, 3 or 4 of the others. If more setups helped, the score would go up with the number.

The two LMT datasets count as one setup.

In [ ]:
units = {"LMT": ["LMT108", "LMT69"], "TUM": ["TUM"], "SENS3": ["SENS3"], "Cluster": ["Cluster"], "Greatest Hits": ["Greatest Hits"]}

if not Path("results/fine_tuning_number_of_setups.csv").exists():
    rows = []
    for target in PUBLIC:
        test = rows_of([target])
        available = [u for u in units if target not in units[u]]
        for n in [1, 2, 3, 4]:
            for chosen in combinations(available, n):
                train = rows_of([d for u in chosen for d in units[u]])
                fine_tune(train)
                rows.append({"target": target, "setups": " + ".join(chosen), "n": n, "score": score(test)})
                pd.DataFrame(rows).to_csv("results/fine_tuning_number_of_setups_partial.csv", index=False)
    pd.DataFrame(rows).to_csv("results/fine_tuning_number_of_setups.csv", index=False)

number = pd.read_csv("results/fine_tuning_number_of_setups.csv")
table = number.groupby(["target", "n"]).score.mean().unstack("n")
table.insert(0, 0, curve[(curve.rate == 2e-5) & (curve.step == 0)].set_index("target").score)
table.loc["mean"] = table.mean()
table.round(2)

In [ ]:
plt.figure(figsize=(6, 3.5))
for target in PUBLIC:
    plt.plot(table.columns, table.loc[target], marker="o", alpha=0.4, label=target)
plt.plot(table.columns, table.loc["mean"], marker="o", color="black", linewidth=2.5, label="mean")
plt.xticks([0, 1, 2, 3, 4], ["original", "1", "2", "3", "4"])
plt.xlabel("number of setups used for fine-tuning")
plt.ylabel("score on the left-out dataset")
plt.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("results/fine_tuning_number_of_setups.png", dpi=120)

## What we see

- On a setup it has never seen, the fine-tuned CLAP is **worse** than the original (0.43 to about 0.31).
- The fall starts in the first steps, so it is not a matter of stopping earlier or training more gently.
- More setups in the training do not help.
- On setups it has seen it is much better (about 0.47 to 0.72).

What the other setups teach is true for them and wrong for the new one.

We also tried, without success (see `RESULTS.md`): editing the fine-tuned weights, a detector that chooses between
the two models, equalizing the frequency bands of each setup, many augmentations, unfreezing the layers one by one.